# Metrics-only pipeline (no model loading)

Recomputes all epistasis metrics from **existing embedding DBs only**. No models are loaded.

**Steps:**
1. Auto-discover which `(source, model)` pairs have DBs under `OUTPUT_BASE`
2. Compute **global** inverse covariance from `COV_INV_SOURCE_NAMES` (cached to `.npz`)
3. Compute **distance-binned** inverse covariance (cached to `.npz`)
4. Recompute metrics with global cov_inv (cached per-model to `.parquet`)
5. Recompute metrics with binned cov_inv (cached per-model to `.parquet`)
6. Merge global + binned into combined parquets

All intermediate results are cached to disk. If the notebook dies, re-run and it picks up where it left off.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 1: Config
# ---------------------------------------------------------------------------
import sys, os, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO)

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "notebooks" / "paper_data_config.py").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from notebooks.processing.pipeline_config import (
    SOURCE_COL, ID_COL, COV_INV_SOURCE_NAMES,
    get_single_dataframe_path,
)
from notebooks.processing.process_epistasis import FULL_MODEL_CONFIG
from notebooks.paper_data_config import EPISTASIS_PAPER_ROOT, data_dir, embeddings_dir

OUTPUT_BASE = embeddings_dir()
SHEETS_DIR = OUTPUT_BASE / "sheets"
CACHE_DIR  = OUTPUT_BASE / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SHEETS_DIR.mkdir(parents=True, exist_ok=True)

# Force recompute even if cache exists? Set True to ignore caches.
FORCE = False

print(f"OUTPUT_BASE = {OUTPUT_BASE}")
print(f"SHEETS_DIR  = {SHEETS_DIR}")
print(f"CACHE_DIR   = {CACHE_DIR}")
print(f"COV_INV_SOURCE_NAMES = {COV_INV_SOURCE_NAMES}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 2: Load dataframe & auto-discover available (source, model) DBs
# ---------------------------------------------------------------------------
import pandas as pd

single_path = get_single_dataframe_path(data_dir)
if single_path is None:
    raise FileNotFoundError("No single dataframe found.")
df_all = pd.read_csv(single_path, sep=None, engine="python")
print(f"Loaded {len(df_all)} rows, sources: {df_all[SOURCE_COL].unique().tolist()}")

# Scan OUTPUT_BASE for all existing .db files
available = {}  # model_key -> set of source_names
all_sources = sorted(d.name for d in OUTPUT_BASE.iterdir() if d.is_dir() and not d.name.startswith(("sheets", "cache", "null_cov", ".")))
for source_name in all_sources:
    source_dir = OUTPUT_BASE / source_name
    for db_file in source_dir.glob("*.db"):
        model_key = db_file.stem
        if model_key in FULL_MODEL_CONFIG:
            available.setdefault(model_key, set()).add(source_name)

MODEL_KEYS = sorted(available.keys())
SOURCE_NAMES = sorted(set().union(*available.values()))

print(f"\nDiscovered {len(MODEL_KEYS)} models across {len(SOURCE_NAMES)} sources:")
for mk in MODEL_KEYS:
    print(f"  {mk}: {sorted(available[mk])}")

# Filter df to only discovered sources
df_all = df_all[df_all[SOURCE_COL].isin(SOURCE_NAMES)].reset_index(drop=True)
print(f"\nFiltered dataframe: {len(df_all)} rows")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 3: Compute global cov_inv (cached to CACHE_DIR/{model_key}_cov_inv.npz)
# ---------------------------------------------------------------------------
import numpy as np
from notebooks.processing.process_epistasis import compute_cov_inv

cov_inv_sources = [s for s in COV_INV_SOURCE_NAMES if s in SOURCE_NAMES]
if not cov_inv_sources:
    raise ValueError(f"None of COV_INV_SOURCE_NAMES={COV_INV_SOURCE_NAMES} found in available sources={SOURCE_NAMES}")

df_cov = df_all[df_all[SOURCE_COL].isin(cov_inv_sources)].reset_index(drop=True)
models_for_cov = [mk for mk in MODEL_KEYS if any(s in available[mk] for s in cov_inv_sources)]

# Check cache
cov_inv_by_model = {}
models_to_compute = []
for mk in models_for_cov:
    cache_path = CACHE_DIR / f"{mk}_cov_inv.npz"
    if cache_path.exists() and not FORCE:
        data = np.load(cache_path)
        cov_inv_by_model[mk] = (data["cov"], data["cov_inv"])
        print(f"  [cache hit] {mk}")
    else:
        models_to_compute.append(mk)

if models_to_compute:
    print(f"Computing global cov_inv for: {models_to_compute}")
    fresh = compute_cov_inv(
        OUTPUT_BASE,
        source_df=df_cov,
        source_col=SOURCE_COL,
        id_col=ID_COL,
        model_keys=models_to_compute,
        method="ledoit_wolf",
        show_progress=True,
    )
    for mk, (cov, cov_inv) in fresh.items():
        cache_path = CACHE_DIR / f"{mk}_cov_inv.npz"
        np.savez(cache_path, cov=cov, cov_inv=cov_inv)
        cov_inv_by_model[mk] = (cov, cov_inv)
        print(f"  [saved] {mk} -> {cache_path.name}")

print(f"\nGlobal cov_inv for {len(cov_inv_by_model)} models: {sorted(cov_inv_by_model.keys())}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 4: Compute distance-binned cov_inv (cached to CACHE_DIR/{model_key}_cov_inv_{bin}.npz)
# ---------------------------------------------------------------------------
from notebooks.processing.process_epistasis import compute_cov_inv_by_distance

BIN_EDGES = (0, 1_000, 10_000, 100_000, float("inf"))
BIN_LABELS = ("0-1kb", "1-10kb", "10-100kb", "100kb+")

# Check cache: one npz per (model, bin)
cov_inv_by_dist = {}  # bin_label -> {model_key: (cov, cov_inv)}
bins_to_compute = set()  # track which models need fresh computation

for mk in models_for_cov:
    all_cached = True
    for label in list(BIN_LABELS) + ["global"]:
        safe_label = label.replace("+", "plus")
        cache_path = CACHE_DIR / f"{mk}_cov_inv_{safe_label}.npz"
        if cache_path.exists() and not FORCE:
            data = np.load(cache_path)
            cov_inv_by_dist.setdefault(label, {})[mk] = (data["cov"], data["cov_inv"])
        else:
            all_cached = False
    if all_cached:
        print(f"  [cache hit] {mk} (all bins)")
    else:
        bins_to_compute.add(mk)

if bins_to_compute:
    models_fresh = sorted(bins_to_compute)
    print(f"Computing binned cov_inv for: {models_fresh}")
    fresh_dist = compute_cov_inv_by_distance(
        OUTPUT_BASE,
        df_cov,
        bin_edges=BIN_EDGES,
        bin_labels=BIN_LABELS,
        source_col=SOURCE_COL,
        id_col=ID_COL,
        model_keys=models_fresh,
        show_progress=True,
    )
    for label, model_dict in fresh_dist.items():
        for mk, (cov, cov_inv) in model_dict.items():
            safe_label = label.replace("+", "plus")
            cache_path = CACHE_DIR / f"{mk}_cov_inv_{safe_label}.npz"
            np.savez(cache_path, cov=cov, cov_inv=cov_inv)
            cov_inv_by_dist.setdefault(label, {})[mk] = (cov, cov_inv)
            print(f"  [saved] {mk} / {label} -> {cache_path.name}")

print(f"\nBinned cov_inv bins: {[k for k in cov_inv_by_dist if k != 'global']}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 5: Recompute metrics with global cov_inv (no model loading, cached per model)
# ---------------------------------------------------------------------------
from genebeddings import VariantEmbeddingDB
from genebeddings.genebeddings import add_epistasis_metrics

metrics_by_model = {}

for model_key in sorted(cov_inv_by_model.keys()):
    cache_path = SHEETS_DIR / f"epistasis_metrics_{model_key}.parquet"
    if cache_path.exists() and not FORCE:
        metrics_by_model[model_key] = pd.read_parquet(cache_path)
        print(f"  [cache hit] {model_key}: {len(metrics_by_model[model_key])} rows")
        continue

    cov, cov_inv = cov_inv_by_model[model_key]
    context = FULL_MODEL_CONFIG[model_key][0]
    sources = df_all[SOURCE_COL].dropna().astype(str).unique().tolist()

    parts = []
    for source_name in sources:
        db_path = OUTPUT_BASE / source_name / f"{model_key}.db"
        if not db_path.exists():
            continue
        df_s = df_all.loc[df_all[SOURCE_COL].astype(str) == source_name].copy()
        if len(df_s) == 0:
            continue
        db = VariantEmbeddingDB(str(db_path))
        try:
            annotated = add_epistasis_metrics(
                df_s, db, model=None, id_col=ID_COL,
                context=context, cov_inv=cov_inv,
                force=False, batch_size=1, show_progress=True,
            )
            parts.append(annotated)
        finally:
            db.close()

    if parts:
        tbl = pd.concat(parts, axis=0).sort_index()
        tbl.to_parquet(cache_path, index=False)
        metrics_by_model[model_key] = tbl
        print(f"  [computed] {model_key}: {len(tbl)} rows -> {cache_path.name}")

print(f"\nGlobal metrics for {len(metrics_by_model)} models")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 6: Recompute metrics with distance-binned cov_inv (no model loading, cached)
# ---------------------------------------------------------------------------
from genebeddings.genebeddings import parse_epistasis_id

# Assign distance bins to all rows
def _get_distance(eid):
    try:
        (_, pos1, _, _, _), (_, pos2, _, _, _) = parse_epistasis_id(str(eid))
        return abs(pos2 - pos1)
    except Exception:
        return float("nan")

distances = df_all[ID_COL].map(_get_distance)
df_all_binned = df_all.copy()
df_all_binned["distance_bin"] = pd.cut(distances, bins=list(BIN_EDGES), labels=list(BIN_LABELS), right=False)

global_cov_inv = cov_inv_by_dist.get("global", {})
model_keys_binned = sorted(global_cov_inv.keys())
metrics_by_model_dist = {}

for model_key in model_keys_binned:
    cache_path = SHEETS_DIR / f"epistasis_metrics_{model_key}_binned.parquet"
    if cache_path.exists() and not FORCE:
        metrics_by_model_dist[model_key] = pd.read_parquet(cache_path)
        print(f"  [cache hit] {model_key}: {len(metrics_by_model_dist[model_key])} rows")
        continue

    context = FULL_MODEL_CONFIG[model_key][0]
    sources = df_all_binned[SOURCE_COL].dropna().astype(str).unique().tolist()
    parts = []

    for source_name in sources:
        db_path = OUTPUT_BASE / source_name / f"{model_key}.db"
        if not db_path.exists():
            continue
        df_s = df_all_binned.loc[df_all_binned[SOURCE_COL].astype(str) == source_name].copy()
        if len(df_s) == 0:
            continue

        db = VariantEmbeddingDB(str(db_path))
        try:
            bin_parts = []
            for label in BIN_LABELS:
                df_bin = df_s.loc[df_s["distance_bin"] == label]
                if len(df_bin) == 0:
                    continue
                # Pick bin-specific cov_inv; fall back to global
                if label in cov_inv_by_dist and model_key in cov_inv_by_dist[label]:
                    _, ci = cov_inv_by_dist[label][model_key]
                elif model_key in global_cov_inv:
                    _, ci = global_cov_inv[model_key]
                else:
                    continue
                annotated_bin = add_epistasis_metrics(
                    df_bin, db, model=None, id_col=ID_COL,
                    context=context, cov_inv=ci,
                    force=False, batch_size=1, show_progress=True,
                )
                bin_parts.append(annotated_bin)

            # NaN distance rows -> use global
            df_nan = df_s.loc[df_s["distance_bin"].isna()]
            if len(df_nan) > 0 and model_key in global_cov_inv:
                _, ci = global_cov_inv[model_key]
                annotated_nan = add_epistasis_metrics(
                    df_nan, db, model=None, id_col=ID_COL,
                    context=context, cov_inv=ci,
                    force=False, batch_size=1, show_progress=True,
                )
                bin_parts.append(annotated_nan)

            if bin_parts:
                parts.append(pd.concat(bin_parts, axis=0))
        finally:
            db.close()

    if parts:
        tbl = pd.concat(parts, axis=0).sort_index()
        tbl.to_parquet(cache_path, index=False)
        metrics_by_model_dist[model_key] = tbl
        print(f"  [computed] {model_key}: {len(tbl)} rows -> {cache_path.name}")

print(f"\nBinned metrics for {len(metrics_by_model_dist)} models")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 7: Merge global + binned into combined parquets
# ---------------------------------------------------------------------------
MAHAL_COLS = ["epi_mahal", "mahal_obs", "mahal_add", "mahal_ratio", "log_mahal_ratio"]

all_model_keys = set(list(metrics_by_model) + list(metrics_by_model_dist))
merged_by_model = {}

for model_key in sorted(all_model_keys):
    df_global = metrics_by_model.get(model_key)
    df_binned = metrics_by_model_dist.get(model_key)

    if df_global is None and df_binned is None:
        continue

    # Start from global: rename mahal cols to global_*
    if df_global is not None:
        base = df_global.copy()
        mahal_present = [c for c in MAHAL_COLS if c in base.columns]
        base = base.rename(columns={c: f"global_{c}" for c in mahal_present})
    else:
        base = df_binned.drop(columns=MAHAL_COLS + ["distance_bin"], errors="ignore").copy()

    # Add binned mahal cols + distance_bin
    if df_binned is not None:
        mahal_present_b = [c for c in MAHAL_COLS if c in df_binned.columns]
        binned_rename = {c: f"binned_{c}" for c in mahal_present_b}
        cols_to_add = list(binned_rename.values()) + ["distance_bin"]
        df_b = df_binned.rename(columns=binned_rename)
        for col in cols_to_add:
            if col in df_b.columns:
                base[col] = df_b[col].values

    merged_by_model[model_key] = base

# Save combined
for model_key, tbl in merged_by_model.items():
    out = SHEETS_DIR / f"epistasis_metrics_{model_key}_combined.parquet"
    tbl.to_parquet(out, index=False)

    raw_cols = [c for c in tbl.columns if not c.startswith(("global_", "binned_")) and c != "distance_bin"]
    global_cols = [c for c in tbl.columns if c.startswith("global_")]
    binned_cols = [c for c in tbl.columns if c.startswith("binned_")]
    print(f"  {model_key}: {tbl.shape} -> {out.name}")
    print(f"    Raw: {len(raw_cols)}, Global mahal: {len(global_cols)}, Binned mahal: {len(binned_cols)}")

print(f"\nDone! {len(merged_by_model)} combined parquets saved to {SHEETS_DIR}")